# 3. Beyond a single train/test split

A model comparison based on one random split can overstate a lucky result. This lab repeats the classical-versus-quantum benchmark on paired splits and summarizes the variation without hiding the underlying runs.

## Run a paired study

Both models see the same data for each seed. Consecutive seeds change dataset generation and the stratified split, while sample count, noise, scaling, and feature-map depth remain fixed.

In [ ]:
from qml_qiskit import run_study

In [ ]:
study = run_study(
    samples=40,
    noise=0.12,
    test_size=0.25,
    base_seed=42,
    runs=5,
    feature_map_reps=2,
)

print("Seeds:", study.seeds)
print(
    "Classical test accuracy: "
    f"{study.classical.test_accuracy_mean:.3f} ± "
    f"{study.classical.test_accuracy_std:.3f}"
)
print(
    "Quantum test accuracy:   "
    f"{study.quantum.test_accuracy_mean:.3f} ± "
    f"{study.quantum.test_accuracy_std:.3f}"
)
print(
    "Paired outcomes (quantum / tie / classical):",
    study.quantum_wins,
    study.ties,
    study.classical_wins,
)

## Audit every run

Aggregate statistics are only trustworthy when their inputs remain inspectable. The complete `BenchmarkResult` objects are stored in `study.benchmarks`.

In [ ]:
print(f"{'Seed':>4} {'Classical':>10} {'Quantum':>10} {'Delta':>9}")
for benchmark in study.benchmarks:
    print(
        f"{benchmark.seed:>4} "
        f"{benchmark.classical.test_accuracy:>10.3f} "
        f"{benchmark.quantum.test_accuracy:>10.3f} "
        f"{benchmark.quantum_advantage:>+9.3f}"
    )

## Verify the summary

These assertions make the core invariants explicit: every run has one paired outcome and reported accuracies stay in their valid range.

In [ ]:
assert study.quantum_wins + study.ties + study.classical_wins == len(study.seeds)
assert 0 <= study.classical.test_accuracy_mean <= 1
assert 0 <= study.quantum.test_accuracy_mean <= 1
assert len(study.benchmarks) == len(study.seeds)

## Interpret carefully

Mean and standard deviation are more informative than a single score, but five small simulated runs still do not establish statistical significance or practical quantum advantage. Increase `runs`, preserve paired splits, and consider confidence intervals or a paired statistical test for a formal study.

## Try it yourself

Repeat the study with feature-map depths 1, 2, and 3. Compare not only accuracy but also mean fit time and support-vector count. Does a deeper circuit improve results consistently across seeds?